### Configurações Gerais

In [1]:
import pandas as pd
import numpy as np
import statsmodels.stats.api as sms
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

### Leitura Inicial

In [2]:
# Caminho da Base
caminho = r'C:\Users\Admin\testeab\testeab\base\ab_data.csv'

In [3]:
# Leitura e armazenamento do dataframe em uma váriavel
df = pd.read_csv(caminho)

In [4]:
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [5]:
print(df.shape) # Quantidade de Linhas e Colunas
print('---')
print(df.columns) # Colunas do dataframe
print('---')
print(df.info()) # Resumo dos dados do dataframe

(294480, 5)
---
Index(['user_id', 'timestamp', 'group', 'landing_page', 'converted'], dtype='object')
---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294480 entries, 0 to 294479
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294480 non-null  int64 
 1   timestamp     294480 non-null  object
 2   group         294480 non-null  object
 3   landing_page  294480 non-null  object
 4   converted     294480 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB
None


In [6]:
print(df.describe()) # Gera um resumo de estatística descritiva

             user_id      converted
count  294480.000000  294480.000000
mean   787973.538896       0.119658
std     91210.917091       0.324562
min    630000.000000       0.000000
25%    709031.750000       0.000000
50%    787932.500000       0.000000
75%    866911.250000       0.000000
max    945999.000000       1.000000


In [7]:
print(df.isnull().sum()) # Quantidade de valores nulos em cada coluna

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64


In [8]:
print(df.duplicated('user_id').sum()) # Verificando se os usuários foram duplicados

3895


In [9]:
print(df.groupby(['group', 'landing_page']).size().reset_index(name='quantidade'))
# Agrupa os dados por grupo de experimento e página de destino, conta os registros e renomeia a coluna com o total

       group landing_page  quantidade
0    control     new_page        1928
1    control     old_page      145274
2  treatment     new_page      145313
3  treatment     old_page        1965


Informações que adquirimos com a leitura inicial dos dados:
* Descobrimos que o dataframe possui 294.480 linhas e 5 colunas;
* Temos informações sobre as colunas;
    * user_id - identificador único do usuário;
    * timestamp -  uma informação de timestamp parcial, representando horário;
    * group - grupo do experimento A/B ao qual o usuário foi atribuído (control ou treatment);
    * landing_page - a página que o usuário viu (old_page para controle ou new_page para tratamento);
    * converted - variável binária indicando se o usuário converteu (1) ou não (0) naquela sessão.
* Temos usuários duplicados;
* Vimos que existe 2 grupos:
    * Grupo de Controle (control) e está associado a landing_page 'old_page';
    * Grupo de Tratamento (treatment) e está associado a landing_page 'new_page'.

Porém observamos que possuimos pessoas do grupo de controle que acessaram a 'new_page' e
pessoas do grupo de tratamento que acessaram a 'old_page'. Para que o teste A/B seja feito sem influencia,
isso indica um erro na implementação do teste e é necessário remover essas divergenciais. Para o teste A/B
é necessário condições de independência, ou seja, o resultado de um usuário tem que ser único e não pode influencia outro.
No caso temos um grupo acessando mais de uma landing_page, o que pode comprometer o teste.

Também é necessário verificar se a questão dos usuários duplicados é por estar acessando ambas as páginas do teste, ou é um problema diferente.
Precisamos de usuários únicos na base.

### Limpeza dos Dados

In [10]:
# 1. Remover linhas onde group e landing_page não correspondem (inconsistentes)
condition = ((df['group'] == 'treatment') & (df['landing_page'] != 'new_page')) | \
            ((df['group'] == 'control') & (df['landing_page'] != 'old_page'))
df_clean = df[~condition].copy()

In [11]:
# 2. Remover usuários duplicados, mantendo apenas a primeira ocorrência de cada user_id
df_clean = df_clean.drop_duplicates(subset='user_id')

In [12]:
# Aplicar limpeza de dados
df_clean.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [13]:
print(df_clean.duplicated('user_id').sum()) # Verificando se os usuários foram duplicados

0


In [14]:
print(df_clean.groupby(['group', 'landing_page']).size().reset_index(name='quantidade'))
# Agrupa os dados por grupo de experimento e página de destino, conta os registros e renomeia a coluna com o total

       group landing_page  quantidade
0    control     old_page      145274
1  treatment     new_page      145311


Agora nossos dados estão consistentes: cada usuário único, atribuído a um único grupo (control ou treatment) e com a página correspondente a seu grupo.

### Análise Exploratoria Inicial (EDA)

In [15]:
# Estatísticas básicas por grupo
group_counts = df_clean['group'].value_counts()
conv_counts = df_clean.groupby('group')['converted'].agg(['count', 'sum', 'mean'])
print("Número de usuários em cada grupo:")
print(group_counts.to_dict())
print("\nConversões e taxa de conversão por grupo:")
print(conv_counts.to_string())

Número de usuários em cada grupo:
{'treatment': 145311, 'control': 145274}

Conversões e taxa de conversão por grupo:
            count    sum      mean
group                             
control    145274  17489  0.120386
treatment  145311  17264  0.118807


Parando para pensar vimos que:
* Percebemos lá no inicio que a função '.describe()' não nos entregou uma boa visão;
* Também não temos uma variedade de dados para verificar para esse estudo;
* Então nosso foco é:
    * Quantos usuários estão em cada grupo?
    * Qual a taxa de conversão em cada grupo?

Com isso pudemos verificar:
* Grupos balanceados:
    * Grupo de Controle com 145.274 usuários;
    * Grupo de Tratamento com 145.311 usuários.
* Taxa de Conversão:
    * Grupo de Controle teve uma taxa de 12,04%;
    * Grupo de Tratamento teve uma taxa de 11,88%.

Em uma comparação "seca", percebemos que a taxa de conversão do grupo de controle é maior (apenas 0,16%),
mas a questão principal do teste a/b é verificar se essa diferença é estatisticamente significativa ou se
pode ser atribuída ao acaso.

Antes de prosseguir, vamos reforçar algumas informações:
Para que o teste A/B seja confiável, a atribuição de usuários aos grupos controle e tratamento deve ser aleatória,
garantindo que não haja vieses sistemáticos (por exemplo, um grupo contendo usuários diferentes em algum aspecto relevante).
Vamos verificar indícios de aleatoriedade:
* Proporção de usuários por grupo: O tamanho dos grupos são quase idênticos (~145 mil cada), o que sugere uma divisão 50/50 bem próxima;
* Usuários únicos: Após a limpeza, cada usuário aparece em apenas um grupo, garantindo independência entre grupos;

Com as informações que temos e após a limpeza dos dados, garantimos que não temos usuários duplicados e que cada grupo vê uma única pagina,
assim reforçamos os indícios de aleatoriedade e descartamos eventos raros que podem influenciar o teste.


### Cálculo do Poder Estatístico

Antes de realizar o teste estatístico, é útil avaliar se nosso experimento tem poder estatístico suficiente para detectar a diferença de conversão que consideramos relevante. O poder estatístico é a probabilidade de rejeitar um falso nulo — ou seja, detectar um efeito existente. Geralmente se define um nível de significância (α) e um poder desejado (1 - β). Aqui foi sugerido verificar o poder para detectar um lift de 1% na taxa de conversão, com α = 5% e poder = 80% (β = 20%).

Em outras palavras, queremos saber quantos usuários seriam necessários em cada grupo para detectar uma diferença de 1 ponto percentual na taxa de conversão (por exemplo, de 12% para 13%) com 80% de chance, dado um teste de significância de 5%. Vamos calcular esse tamanho de amostra usando análise de poder para teste de proporções (assumindo grupos de mesmo tamanho):

In [16]:
# Taxa de conversão de base (grupo controle) estimada:
p_control = df_clean[df_clean['group']=='control']['converted'].mean()  # ~0.1204
# Supor um aumento absoluto de 1% na taxa de conversão para o tratamento:
p_treatment = p_control + 0.01  # por exemplo, ~0.1304 ou 13.04%

# Calcular o efeito (effect size) em termos de proporções (Cohen's h)
effect_size = proportion_effectsize(p_control, p_treatment)

# Inicializar análise de poder para duas proporções independentes
analysis = sms.NormalIndPower()
# Calcular n (tamanho por grupo) necessário para detectar esse efeito com 80% poder e 5% de significância (bicaudal)
required_n = analysis.solve_power(effect_size=effect_size, power=0.80, alpha=0.05, ratio=1)
print(f"Taxa de conversão base: {p_control*100:.2f}%")
print(f"Taxa de conversão com 1% de lift: {p_treatment*100:.2f}%")
print(f"Tamanho de amostra necessário por grupo (~80% poder, 5% sig.): {required_n:.0f} usuários por grupo")


Taxa de conversão base: 12.04%
Taxa de conversão com 1% de lift: 13.04%
Tamanho de amostra necessário por grupo (~80% poder, 5% sig.): 17209 usuários por grupo


Este cálculo indica que, para detectar uma diferença absoluta de 1,0 ponto percentual (de 12,04% para 13,04% de conversão) com 80% de probabilidade, seriam necessários aproximadamente 17.209 usuários em cada grupo (ou cerca de 34.418 usuários no total). 

Comparando com nosso experimento atual, que tem ~145 mil usuários em cada grupo, vemos que a amostragem do teste é muito maior do que o mínimo necessário para esse efeito. Isso significa que o teste atual tem um poder estatístico bem alto para detectar até mesmo pequenos lifts de 1%. Na prática, com 145 mil usuários por grupo, o poder para detectar um lift de 1% seria próximo de 100%. Portanto, se houvesse um aumento verdadeiro de 1% (ou mais) na conversão devido à página nova, é muito provável que nosso teste o detectasse de forma significativa.

### Teste Estatístico (Teste Z de Proporção)

O teste A/B é um teste de hipotese, então vamos considerar 2 cenários e tentar comprovar qual deles é verdadeiro:
* Cenário 1: As taxas de conversão do grupo tratamento e do grupo controle são iguais. Ou seja, qualquer diferença observada é devida ao acaso.
* Cenário 2: A taxa de conversão do grupo tratamento é diferente da do grupo controle.

Na literatura usamos H0 (Cenário 1) e H1 (Cenário 2).

Vamos fazer o Z-teste para obtermos o p-valor.

Usamos o p-valor na prática para saber se devemos rejeitar ou não H0 (Cenário 1), ou seja,
entender se nossos dados amostrais são tão diferentes do que dissemos na hipótese nula a ponto de rejeitar H0

O p-valor é justamente a probabilidade de obtermos o dado que medimos na amostra quando considerarmos o cenário ficticio que H0 é verdadeiro.
Se for muito pequeno, então rejeitamos H0.

O valor de tolerancia é 5%. Ou seja, se o p-valor for menor que 5% rejeitamos H0.

In [ ]:
# Contagens de conversões em cada grupo:
convert_control = df_clean.query("group == 'control' and converted == 1").shape[0]
convert_treatment = df_clean.query("group == 'treatment' and converted == 1").shape[0]
n_control = df_clean.query("group == 'control'").shape[0]
n_treatment = df_clean.query("group == 'treatment'").shape[0]

# Aplicar teste Z para diferença de proporções
count = np.array([convert_treatment, convert_control])
nobs = np.array([n_treatment, n_control])
z_stat, pval_two_sided = proportions_ztest(count, nobs, alternative="two-sided")
print(f"z_estatística = {z_stat:.4f}, p-valor (bicaudal) = {pval_two_sided:.4f}")

z_estatística = -1.3116, p-valor (bicaudal) = 0.1897


Obtivemos o P-Valor de 0.1897 (18,97%).

Os dados não suportam a afirmação de que a nova página (tratamento) aumenta a taxa de conversão em relação à página antiga (controle).

Qualquer pequena diferença observada no nosso experimento pode ser atribuída ao acaso, dado o alto valor-p obtido.

## Informações sobre o Z-Test

### Z-Test para Proporções (frequentista)

### Vantagens:

- Rápido e simples de aplicar
- Muito usado para comparar **proporções** (como taxas de conversão)
- Teoricamente bem fundamentado

### Desvantagens:

- **Assume distribuição normal** (aproximação via Teorema Central do Limite)
- Precisa de **n ≥ 30 por grupo** (você tem muito mais, então tá tranquilo)
- Depende de condições como independência e variância homogênea

### Quando usar:

> Quando você tem grandes amostras e quer uma resposta rápida e objetiva sobre a diferença entre grupos.
>

### **1. "Assume distribuição normal" — O que significa?**

O **z-test** é baseado na ideia de que, se você tem uma **amostra grande o suficiente**, a **distribuição da média amostral (ou da diferença entre proporções)** vai se aproximar de uma **distribuição normal (curva em forma de sino)**.

#### Por quê?

Por causa do **Teorema Central do Limite (TCL)**:

> Mesmo que os dados originais não sejam normalmente distribuídos, a média de muitas amostras grandes será aproximadamente normal.
> 

#### Exemplo:

Você está observando taxas de conversão (0 ou 1). Isso **não é normal** (é binário).

Mas se você calcular a média de conversão de 100 mil usuários do grupo "control", isso **sim** tende à normal.

Por isso podemos usar o z-test com proporções mesmo que os dados sejam binários — **se o tamanho da amostra for grande**.

---

### **2. Como sei se tenho `n >= 30` por grupo?**

Aqui é simples! Você só precisa contar quantos registros tem em cada grupo.

Se cada grupo (control e treatment) tiver pelo menos 30 registros, está OK. No seu caso:

- `control`: 145.274 registros
- `treatment`: 145.313 registros

**Muito acima de 30!** Ou seja, **o z-test pode ser aplicado com tranquilidade**.

---

### **3. Condições do z-test: Independência e Variância homogênea**

#### a) Independência

Isso significa que **o resultado de um usuário não influencia o outro**.

✅ No seu caso:

- Os usuários são únicos (checamos isso)
- Cada um viu apenas uma página
- Eles não interagem entre si

Então podemos dizer que os dados **são independentes**.

---

#### b) Variância homogênea (ou homocedasticidade)

Para o z-test, assumimos que a **variância da proporção é razoavelmente parecida entre os dois grupos**.

Isso se traduz na fórmula da variância da proporção:

Var(p)=p(1−p)nVar(p) = \frac{p(1-p)}{n}

Var(p)=np(1−p)

Se as **taxas de conversão** forem próximas e os **tamanhos das amostras** forem similares, então as variâncias também serão parecidas.

✅ No seu caso:

- `control`: 12,04%
- `treatment`: 11,88%
- Tamanhos quase iguais

Ou seja, **a variância pode ser considerada homogênea** aqui.